In [0]:
DESCRIBE HISTORY retail_lakehouse.silver.customers;

**Simulating incremental update**

In [0]:
-- Create checkpoint table to track last processed version
CREATE TABLE IF NOT EXISTS retail_lakehouse.gold.scd2_checkpoint (
    table_name STRING,
    last_processed_version BIGINT,
    last_updated TIMESTAMP
);

-- Initialize with starting version if not exists
MERGE INTO retail_lakehouse.gold.scd2_checkpoint AS tgt
USING (SELECT 'retail_lakehouse.silver.customers' AS table_name) AS src
ON tgt.table_name = src.table_name
WHEN NOT MATCHED THEN
    INSERT (table_name, last_processed_version, last_updated)
    VALUES (src.table_name, 7, CURRENT_TIMESTAMP());

In [0]:
-- Test with another customer update
UPDATE retail_lakehouse.silver.customers
SET City = 'Chennai', Address = 'Anna Nagar'
WHERE CustomerID = 2;

In [0]:

UPDATE retail_lakehouse.silver.customers
SET City = 'Hyderabad'
WHERE CustomerID = 1;


In [0]:
-- Get last processed version from checkpoint
DECLARE OR REPLACE VARIABLE last_version BIGINT;
SET VAR last_version = (
    SELECT last_processed_version 
    FROM retail_lakehouse.gold.scd2_checkpoint 
    WHERE table_name = 'retail_lakehouse.silver.customers'
);

-- View changes since last checkpoint
SELECT
    _change_type,
    _commit_version,
    CustomerID,
    CustomerName,
    City
FROM table_changes(
    'retail_lakehouse.silver.customers',
    last_version
)
WHERE _change_type IN ('update_preimage', 'update_postimage')
ORDER BY _commit_version DESC;

In [0]:
-- EXPIRE OLD ACTIVE RECORD

MERGE INTO retail_lakehouse.gold.dim_customer tgt
USING (
    SELECT DISTINCT
        CustomerID,
        City,
        Address,
        Email
    FROM table_changes(
        'retail_lakehouse.silver.customers',
        last_version
    )
    WHERE _change_type = 'update_postimage'
) src
ON tgt.CustomerID = src.CustomerID
    AND tgt.IsActive = TRUE
WHEN MATCHED
    AND (
        tgt.City <> src.City
        OR tgt.Address <> src.Address
        OR tgt.Email <> src.Email
    )
THEN UPDATE SET
    tgt.EndDate = CURRENT_DATE(),
    tgt.IsActive = FALSE;

In [0]:
-- INSERT NEW ACTIVE RECORD

INSERT INTO retail_lakehouse.gold.dim_customer
(
    CustomerID,
    CustomerName,
    Email,
    City,
    Address,
    StartDate,
    EndDate,
    IsActive
)
SELECT
    src.CustomerID,
    src.CustomerName,
    src.Email,
    src.City,
    src.Address,
    CURRENT_DATE() AS StartDate,
    DATE '9999-12-31' AS EndDate,
    TRUE AS IsActive
FROM (
    SELECT DISTINCT
        CustomerID,
        CustomerName,
        Email,
        City,
        Address
    FROM table_changes(
        'retail_lakehouse.silver.customers',
        last_version
    )
    WHERE _change_type = 'update_postimage'
) src
LEFT JOIN retail_lakehouse.gold.dim_customer tgt
    ON src.CustomerID = tgt.CustomerID
    AND tgt.IsActive = TRUE
    AND src.City = tgt.City
    AND src.Address = tgt.Address
    AND src.Email = tgt.Email
WHERE tgt.CustomerID IS NULL;

In [0]:
-- CLEANUP: Remove duplicate active records keeping only the most recent one
-- This fixes existing duplicates in the table

MERGE INTO retail_lakehouse.gold.dim_customer AS tgt
USING (
    SELECT 
        CustomerSK,
        CustomerID,
        ROW_NUMBER() OVER (
            PARTITION BY CustomerID 
            ORDER BY CustomerSK DESC
        ) AS rn
    FROM retail_lakehouse.gold.dim_customer
    WHERE IsActive = TRUE
    QUALIFY rn > 1  -- Keep only duplicates (not the most recent)
) dup
ON tgt.CustomerSK = dup.CustomerSK
WHEN MATCHED THEN
    UPDATE SET
        tgt.EndDate = CURRENT_DATE(),
        tgt.IsActive = FALSE;

-- Verify cleanup
SELECT 
    CustomerID,
    COUNT(*) as active_count
FROM retail_lakehouse.gold.dim_customer
WHERE IsActive = TRUE
GROUP BY CustomerID
HAVING COUNT(*) > 1;

In [0]:
-- Update checkpoint to latest processed version
DECLARE OR REPLACE VARIABLE checkpoint_last_version BIGINT;
SET VAR checkpoint_last_version = (
    SELECT last_processed_version 
    FROM retail_lakehouse.gold.scd2_checkpoint 
    WHERE table_name = 'retail_lakehouse.silver.customers'
);

UPDATE retail_lakehouse.gold.scd2_checkpoint
SET 
    last_processed_version = (
        SELECT MAX(_commit_version) 
        FROM table_changes(
            'retail_lakehouse.silver.customers',
            checkpoint_last_version
        )
    ),
    last_updated = CURRENT_TIMESTAMP()
WHERE table_name = 'retail_lakehouse.silver.customers';

-- Verify checkpoint updated
SELECT * FROM retail_lakehouse.gold.scd2_checkpoint;

In [0]:
-- VALIDATE SCD TYPE 2

SELECT
    CustomerID,
    CustomerName,
    City,
    StartDate,
    EndDate,
    IsActive
FROM retail_lakehouse.gold.dim_customer
WHERE CustomerID = 1
ORDER BY StartDate;

-- =========================================
-- EXPECTED RESULT
-- =========================================

-- OLD RECORD
-- IsActive = FALSE
-- EndDate updated

-- NEW RECORD
-- IsActive = TRUE
-- StartDate = current_date

In [0]:

-- ACTIVE vs INACTIVE VALIDATION

SELECT
    CustomerID,
    City,
    StartDate,
    EndDate,
    IsActive
FROM retail_lakehouse.gold.dim_customer
WHERE CustomerID = 1;

In [0]:
--HISTORICAL DATA VALIDATION
SELECT *
FROM retail_lakehouse.gold.dim_customer
WHERE CustomerID = 1;

-- VERIFY ONLY ONE ACTIVE RECORD
SELECT
    CustomerID,
    COUNT(*)
FROM retail_lakehouse.gold.dim_customer
WHERE IsActive = TRUE
GROUP BY CustomerID
HAVING COUNT(*) > 1;

-- =========================================
-- EXPECTED RESULT:
-- 0 rows
-- =========================================

-- If this returns multiple rows, it means there are multiple active records (IsActive = TRUE) for the same CustomerID.
-- This can happen if the SCD Type 2 logic did not properly expire the old record (set IsActive = FALSE) before inserting the new one.
-- Check for records with CustomerID = 1 and IsActive = TRUE to investigate:

SELECT *
FROM retail_lakehouse.gold.dim_customer
WHERE CustomerID = 1 AND IsActive = FALSE;